In [1]:
# MOM
# build a system which generates the MOM from the transcript of the meeting using RAG

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain.chat_models import init_chat_model

# -------------------------------
# Init LLM (Groq)
# -------------------------------
llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
)

# -------------------------------
# Sample Transcript
# -------------------------------
transcript = """
Ronish: We need to finalize the API by Friday.
Manager: Deadline is strict.
Dev1: Database schema is ready.
Manager: Ronish will handle API. Dev1 will review schema.
Dev2: There might be delay due to vendor dependency.
"""

# -------------------------------
# Chunking
# -------------------------------
def chunk_text(text, size=2):
    lines = text.strip().split("\n")
    print("lines >>>>>>>>>>>",lines)
    return [" ".join(lines[i:i+size]) for i in range(0, len(lines), size)]

chunks = chunk_text(transcript)
print("chunks >>>>>>>>>>>",chunks)
# -------------------------------
# Embedding + FAISS
# -------------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)
print("embeddings >>>>>>>>>>>",embeddings)
index = faiss.IndexFlatL2(embeddings.shape[1])
print("index >>>>>>>>>>>",index)
index.add(np.array(embeddings))
print("index 1 >>>>>>>>>>>",index)
# -------------------------------
# Retrieval
# -------------------------------
def retrieve(query, top_k=3):
    q_emb = model.encode([query])
    _, idx = index.search(np.array(q_emb), top_k)
    return [chunks[i] for i in idx[0]]

# -------------------------------
# Generate MoM
# -------------------------------
def generate_mom():
    context = retrieve("summary action items decisions risks")
    context_text = "\n".join(context)

    prompt = f"""
Generate Minutes of Meeting (MoM).

Include:
1. Summary
2. Action Items (with owner)
3. Decisions
4. Risks

Transcript:
{context_text}
"""

    response = llm.invoke(prompt)
    return response.content

# -------------------------------
# Run
# -------------------------------
print(generate_mom())

E:\EDU_CARE\arg_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\EDU_CARE\arg_venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


lines >>>>>>>>>>> ['Ronish: We need to finalize the API by Friday.', 'Manager: Deadline is strict.', 'Dev1: Database schema is ready.', 'Manager: Ronish will handle API. Dev1 will review schema.', 'Dev2: There might be delay due to vendor dependency.']
chunks >>>>>>>>>>> ['Ronish: We need to finalize the API by Friday. Manager: Deadline is strict.', 'Dev1: Database schema is ready. Manager: Ronish will handle API. Dev1 will review schema.', 'Dev2: There might be delay due to vendor dependency.']
embeddings >>>>>>>>>>> [[-0.06814667  0.02156823  0.02344047 ...  0.04406067 -0.01777767
   0.04295603]
 [-0.00745296 -0.07260253 -0.05365206 ...  0.05801018  0.06982605
  -0.00609497]
 [-0.0178346  -0.08701435  0.04551784 ... -0.05210252  0.07486763
  -0.00599302]]
index >>>>>>>>>>> <faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000029A7B84B540> >
index 1 >>>>>>>>>>> <faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFla

In [2]:
# Day 12 POC1 - LangChain Version
# 2 agents with different sources, supervisor routes the query to the correct agent

from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

# Initialize LLM
llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
)

# Prompt Templates

supervisor_prompt = PromptTemplate(
    input_variables=["query"],
    template="""
You are a Supervisor assistant.
You need to choose either hr_agent or it_agent.
Select the correct agent only from the User question.
Reply with ONLY one of these two words: hr_agent or it_agent
No explanation.

Question: {query}
"""
)

hr_prompt = PromptTemplate(
    input_variables=["query"],
    template="""
You are an HR support assistant.
Answer the employee question clearly.

Question: {query}
"""
)

it_prompt = PromptTemplate(
    input_variables=["query"],
    template="""
You are an IT support assistant.
Help fix the technical issue.

Question: {query}
"""
)

# Chains (LCEL)

supervisor_chain = supervisor_prompt | llm
hr_chain         = hr_prompt         | llm
it_chain         = it_prompt         | llm

# Agent Functions

def supervisor_node(query: str) -> str:
    print("supervisor_node got query:", query)
    response = supervisor_chain.invoke({"query": query})
    selected_agent = response.content.strip().lower()
    print("Selected agent is >>>>>>>>>>>>>>>>>> ", selected_agent)
    return selected_agent


def hr_agent_node(query: str) -> str:
    response = hr_chain.invoke({"query": query})
    return response.content


def it_agent_node(query: str) -> str:
    response = it_chain.invoke({"query": query})
    return response.content


# Router Logic

def router(selected_agent: str) -> str:
    if "hr" in selected_agent:
        return "hr_agent"
    else:
        return "it_agent"


# Main Runner

def run_agent(query: str) -> str:

    # Step 1: Supervisor decides
    selected_agent = supervisor_node(query)

    # Step 2: Route to correct agent
    agent = router(selected_agent)

    # Step 3: Run the selected agent
    if agent == "hr_agent":
        print("Routing to HR Agent...")
        answer = hr_agent_node(query)

    else:
        print("Routing to IT Agent...")
        answer = it_agent_node(query)

    return answer


# Run Examples

# Example 1 — HR Query
print("\n--- HR Query ---")
result1 = run_agent("How many leave days do I have?")
print("\nHR Agent Response:")
print(result1)

# Example 2 — IT Query
print("\n--- IT Query ---")
result2 = run_agent("My VPN is not working")
print("\nIT Agent Response:")
print(result2)


--- HR Query ---
supervisor_node got query: How many leave days do I have?
Selected agent is >>>>>>>>>>>>>>>>>>  hr_agent
Routing to HR Agent...

HR Agent Response:
To check your leave balance, I'll need to look up your employee information in our HR system. Can you please provide me with your employee ID or your name so I can verify your details?

Once I have that information, I can check your leave balance and let you know how many leave days you have available.

--- IT Query ---
supervisor_node got query: My VPN is not working
Selected agent is >>>>>>>>>>>>>>>>>>  it_agent
Routing to IT Agent...

IT Agent Response:
I'd be happy to help you troubleshoot the issue with your VPN. Can you please provide me with some more information about the problem you're experiencing? 

Here are a few questions to get started:

1. What type of VPN are you using (e.g. Cisco AnyConnect, OpenVPN, etc.)?
2. Have you recently made any changes to your network or VPN settings?
3. Are you getting any error 

In [3]:
# POC 2 - LangChain Version
# Build a system which can convert one programming language code to another

from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

# Initialize Model
llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
)

# Prompt Templates

generate_prompt = PromptTemplate(
    input_variables=["source_lang", "target_lang", "source_code"],
    template="""
Convert the following {source_lang} code into {target_lang} code.

Requirements:
- Must be executable
- Correct syntax
- Include imports if needed
- Only output code, no explanation

Source Code:
{source_code}
"""
)

validate_prompt = PromptTemplate(
    input_variables=["target_lang", "converted_code"],
    template="""
Check if this {target_lang} code is correct and executable.

Code:
{converted_code}

Reply ONLY:
VALID
or
INVALID: <reason>
"""
)

correct_prompt = PromptTemplate(
    input_variables=["target_lang", "validation_result", "converted_code"],
    template="""
Fix this {target_lang} code.

Error:
{validation_result}

Code:
{converted_code}

Return corrected executable code only.
"""
)

# Chains (using LCEL)

generate_chain = generate_prompt | llm
validate_chain = validate_prompt | llm
correct_chain  = correct_prompt  | llm

# Core Logic

def run_code_converter(source_code, source_lang, target_lang, max_retries=3):

    # Step 1: Generate
    print("Generating converted code...")
    response = generate_chain.invoke({
        "source_lang": source_lang,
        "target_lang": target_lang,
        "source_code": source_code
    })
    converted_code = response.content

    retry = 0

    while retry < max_retries:

        # Step 2: Validate
        print(f"\nValidating... (Attempt {retry + 1})")
        val_response = validate_chain.invoke({
            "target_lang": target_lang,
            "converted_code": converted_code
        })
        validation_result = val_response.content
        print("Valid/Invalid >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> ", validation_result)

        # Step 3: Router Logic
        if "VALID" in validation_result:
            print("\nCode is VALID. Stopping.")
            break

        else:
            print("\nCode is INVALID. Attempting correction...")
            # Step 4: Correct
            cor_response = correct_chain.invoke({
                "target_lang": target_lang,
                "validation_result": validation_result,
                "converted_code": converted_code
            })
            converted_code = cor_response.content
            retry += 1

    return converted_code, validation_result


# Run

source_code = """
def add(a, b):
    return a + b

print(add(2,3))
"""

converted_code, validation_result = run_code_converter(
    source_code=source_code,
    source_lang="Python",
    target_lang="Java"
)

# Output

print("\nConverted Code:\n")
print(converted_code)

print("\nValidation Result:\n")
print(validation_result)

Generating converted code...

Validating... (Attempt 1)
Valid/Invalid >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  VALID

Code is VALID. Stopping.

Converted Code:

```java
public class Main {
    public static void main(String[] args) {
        System.out.println(add(2, 3));
    }

    public static int add(int a, int b) {
        return a + b;
    }
}
```

Validation Result:

VALID
